# Virtual Environments Solutions

Solutions for `exercises.ipynb`. Try the exercises first — peek here only after you've attempted each question.

## Part 1 — Warm-up

**1. Why isolate.** Upgrading for B silently breaks A — "dependency hell". One venv per project gives each its own site-packages.

In [ ]:
# Global installs make every upgrade a coin flip: fixing B (numpy 2.x)
# silently breaks A (needs 1.x). Multiply by several projects -> dependency hell.
#
# The cure - a private environment INSIDE the project folder:
#
# cd my_project
# python -m venv .venv

**2. Flip the switch.** Activation points PATH at the venv; success shows as a `(.venv)` prefix on your prompt.

In [ ]:
# Windows PowerShell : .venv\Scripts\Activate.ps1
# Windows CMD        : .venv\Scripts\activate.bat
# Windows Git Bash   : source .venv/Scripts/activate
# macOS / Linux      : source .venv/bin/activate
#
# Leave (everywhere) : deactivate
#
# Success looks like:
# (.venv) C:\Users\fahim\my_project>

**3. Ask Python, not the prompt.** `sys.prefix != sys.base_prefix` is Python-side proof of a venv; terminals can lie.

In [ ]:
import os
import sys

print("executable :", sys.executable)
print("prefix     :", sys.prefix)         # where THIS python looks for packages
print("base       :", sys.base_prefix)   # the real installation
print("in a venv? :", sys.prefix != sys.base_prefix)
print("VIRTUAL_ENV set:", "VIRTUAL_ENV" in os.environ)
# Inside an activated venv: prefix points INTO .venv, differs from base,
# and VIRTUAL_ENV is set. Here (plain interpreter) both checks say False.

## Part 2 — Practice

**4. The standard workflow.** create -> activate -> install -> freeze -> gitignore, strictly in that order.

In [ ]:
# 1. once per project
# cd my_project
# python -m venv .venv
#
# 2. every session
# source .venv/Scripts/activate        # Git Bash (.ps1 / .bat on PS/cmd)
#
# 3. install what the project needs
# python -m pip install requests pandas
#
# 4. snapshot the exact set
# python -m pip freeze > requirements.txt
#
# 5. keep git clean
# echo ".venv/" >> .gitignore

**5. Rebuild it anywhere.** Fresh venv + `install -r` reproduces your setup byte-for-byte; you never need to activate to USE a venv.

In [ ]:
# python -m venv .venv && source .venv/Scripts/activate
# python -m pip install -r requirements.txt

# Shortcut - call the venv's python directly, no activation needed:
# .venv/Scripts/python train.py

**6. Anatomy of .venv.** It's a thin shim over your real Python — seconds to create, disposable by design.

In [ ]:
import venv

print("venv builder available:", hasattr(venv, "EnvBuilder"))
# Inside .venv you find:
#   python.exe / python   -> launcher wired to THIS environment
#   Lib/site-packages/    -> the project's PRIVATE packages
#   Scripts/activate*     -> scripts that switch your terminal in
#   pyvenv.cfg            -> points back to the base interpreter
# NOT a copy: the standard library is shared with the base install,
# which is why creation takes seconds, not minutes.

## Part 3 — Challenge

**7. The honest detector.** `sys.prefix != sys.base_prefix` cannot lie — unlike a prompt that may belong to a stale or unactivated shell.

In [ ]:
import sys


def in_venv():
    """True only when running inside a virtual environment."""
    return sys.prefix != sys.base_prefix


def report():
    print("executable:", sys.executable)
    print("prefix    :", sys.prefix)
    print("base      :", sys.base_prefix)
    print("verdict   :", "inside a venv" if in_venv() else "base/global Python")


report()
# This reports the truth of whatever interpreter executes the code -
# a stale prompt (or forgotten activation) can't fool sys.prefix.

**8. Three broken environments.** Match the activation script to the shell; treat venvs as disposable and rebuild from requirements.txt; unactivated installs land in the GLOBAL Python.

In [ ]:
# (a) Wrong script for the shell. In cmd use:  .venv\Scripts\activate.bat
#     (PowerShell blocked? Fix the execution policy ONCE per user:
#      Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope CurrentUser)
#
# (b) venv paths are baked in, so moving/renaming kills it. Treat the
#     folder as disposable - delete .venv, recreate, reinstall:
#       python -m venv .venv
#       python -m pip install -r requirements.txt
#
# (c) Commit the RECIPE, not the kitchen: add ".venv/" to .gitignore,
#     remove the folder from git tracking, and ship requirements.txt.
#
# Bonus: with no activation, packages land in the GLOBAL Python -
# check the (.venv) prompt, or better, run the sys.prefix check first.